# M2 çalışma verisi — kalite kontrolleri
## tl;dr
15 çalışma-kapsamlı hammadde adayı; 0 motor kabulü. İki S0 serisindeki 12 TEC aralık ortalaması fizik motoruna girdi değildir. Tamlık, sayımlar ve değişmez arşivden tekrar üretim denetlenir; bağımsız deney doğrulaması yapılmaz.
## Context & Methods
### Key Assumptions
Kaynak: data/reference/study-material-candidates-v1.json ve onun DOI/URL/hash kayıtları. Grain: çalışma içindeki malzeme analizi. Yayın tarihi analiz tarihi değildir. Sayısal sütun toplamı yalnızca inceleme alarmıdır; eksik hücre sıfır değildir. Zaman serisi/trend değerlendirmesi için uygun veri yok. Grafik bu küçük bütünlük kontrolüne ek bilgi sağlamaz.
Gereksinim: Python 3.12, nbformat/nbclient/ipykernel; son hücre için yerel storage/research arşivi. Ağ çağrısı yok.

In [ ]:
from pathlib import Path
from collections import Counter
from decimal import Decimal
import json, hashlib, sys
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / 'data/reference/study-material-candidates-v1.json').is_file())
sys.path.insert(0, str(root))
path = root / 'data/reference/study-material-candidates-v1.json'
data = json.loads(path.read_text(encoding='utf-8'))
records = data['records']
print('Aday:', len(records), '| Motor kabulü:', sum(r['core_engine_eligible'] for r in records))
print('SHA256:', hashlib.sha256(path.read_bytes()).hexdigest())

## Results
Kimlik, kaynak bağlantısı ve eksik bilgi kontrolleri; tarih/lot/baz boşlukları tamamlanmış gibi gösterilmez.

In [ ]:
assert len({r['id'] for r in records}) == len(records)
assert all(r['source_id'] in data['sources'] for r in records)
assert all(s['source_author'] and s['source_license'] == 'CC-BY-4.0' for s in data['sources'].values())
assert all(not r['core_engine_eligible'] and r['disposition'] == 'QUARANTINED' for r in records)
print('Kaynak başına aday:', dict(Counter(r['source_id'] for r in records)))
print('Bazı bilinmeyen:', sum(r['analysis_basis'] == 'UNKNOWN' for r in records))
print('wt% LOI kullanılamayan:', sum('LOI_WT_PCT_UNAVAILABLE' in r['quality_flags'] for r in records))
print('Kimlik ve kaynak kontrolleri: PASS')

In [ ]:
for r in records:
    total = sum((Decimal(m['value']) for m in r['measurements'] if m['value'] is not None), Decimal(0))
    assert total == Decimal(r['reported_numeric_column_sum'])
    if abs(total - 100) > Decimal('0.5'):
        print(r['name_as_reported'], ':', total, '(düzeltilmedi)')
thermal = data['thermal_references']
assert all(t['quantity_kind'] == 'REPORTED_INTERVAL_MEAN_LINEAR_TEC' and not t['model_input_eligible'] for t in thermal)
print('Termal seri:', len(thermal), '| Aralık değeri:', sum(len(t['intervals']) for t in thermal))

In [ ]:
from pipelines.ingestion.study_materials import build
assert data == build(root / 'storage/research')
print('Yerel arşivden tam replay: PASS')

## Takeaways
Bu kayıtlar izlenebilir araştırma adaylarıdır, güncel üretici lot sertifikası değildir. Analiz bazı ve eksik kimya açıklanmadan çekirdeğe kabul yok. Dört sütun toplamı inceleme bandı dışında; nedenini belirlemek için kaynak teyidi gerekir. TEC ortalamaları anlık genleşme eğrisine çevrilmedi. Ayrıntı: docs/M2_BATCH3_REPORT.md; gönderilmemiş sorular: docs/REFERENCE_DATA_REQUEST.md.